## Model Selection: DeBERTa v3 

### Preparation

In [1]:
!pip uninstall -y transformers peft trl bitsandbytes
!pip install -q -U transformers datasets scikit-learn accelerate

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 10.5 MB/s eta 0:00:00


## Execution

In [3]:
import os
import sys
import gc
import torch
if not torch.cuda.is_available():
    raise SystemError("NO GPU DETECTED! Go to Runtime -> Change runtime type -> Select T4 GPU.")
import random
import numpy as np
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score, classification_report
from huggingface_hub import login

# ---------------------------------------------------------
# 1. REPRODUCIBILITY & MEMORY CLEANSING
# ---------------------------------------------------------
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

if 'trainer' in globals():
    del trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU memory cleared. Seeds locked.")

# ---------------------------------------------------------
# 2. ENVIRONMENT CONFIGURATION
# ---------------------------------------------------------
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

    REPO_DIR = "/content/drive/MyDrive/progettoLLM/CLARITY"
    hf_cache_dir = "/content/drive/MyDrive/progettoLLM/hf_cache"
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir

    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Colab environment and Google Drive configured. Model cache at: {hf_cache_dir}")
except ImportError:
    print("Local environment detected. Default Hugging Face cache will be used.")

# Authentication Token Management
hf_token = None
try:
    from google.colab import userdata
    print("Acquiring token from Google Secrets...")
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    print("Reading token from local .env file...")
    env_path = ".env"
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    break

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print("Hugging Face authentication completed.")
else:
    print("Warning: HF token not found. Possible errors during model download.")

# ---------------------------------------------------------
# 3. LABEL MAPPING & TOKENIZER SETUP
# ---------------------------------------------------------
model_id = "microsoft/deberta-v3-base"

# Mapping the 3 main Clarity classes
label2id = {
    "Clear Reply": 0,
    "Ambivalent": 1,
    "Clear Non-Reply": 2
}
id2label = {v: k for k, v in label2id.items()}

print("Loading Tokenizer...")
# DeBERTa v3 uses a sentencepiece tokenizer, fast and efficient
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

# ---------------------------------------------------------
# 4. DATASET LOADING & TOKENIZATION
# ---------------------------------------------------------
train_dataset = load_from_disk("dataset/QEvasion/train")
test_dataset = load_from_disk("dataset/QEvasion/test")

def tokenize_function(example):
    # Encoders read the whole context bidirectionally.
    # We feed it the question and answer separated by the tokenizer's native SEP token.
    text = f"Question: {example['question']} {tokenizer.sep_token} Answer: {example['interview_answer']}"

    tokenized_inputs = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=512 # Standard limit for DeBERTa base
    )

    # THE FIX IS HERE: example['clarity_label'] is now correctly handled as a single string
    label_str = example['clarity_label']
    tokenized_inputs["label"] = label2id.get(label_str, -1)

    return tokenized_inputs

print("Tokenizing datasets...")
# THE SECOND HALF OF THE FIX: Removed batched=True
train_dataset = train_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

# Filter out any unmapped labels (-1)
train_dataset = train_dataset.filter(lambda x: x["label"] != -1)
test_dataset = test_dataset.filter(lambda x: x["label"] != -1)

# Remove unused text columns so the PyTorch DataLoader doesn't crash
columns_to_remove = ["question", "interview_answer", "clarity_label", "evasion_label"]
for col in train_dataset.column_names:
    if col not in ["input_ids", "attention_mask", "label"]:
        columns_to_remove.append(col)

train_dataset = train_dataset.remove_columns(list(set(columns_to_remove)))
test_dataset = test_dataset.remove_columns(list(set(columns_to_remove)))

# ---------------------------------------------------------
# 5. MODEL INITIALIZATION
# ---------------------------------------------------------
print("Loading Model...")
# Standard FP32 loading. No QLoRA, no BitsAndBytes needed.
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

# ---------------------------------------------------------
# 6. TRAINING CONFIGURATION
# ---------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./deberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,

    # DeBERTa fits easily on a T4, so we can use larger batches
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,

    learning_rate=2e-5, # Lower learning rate for full fine-tuning
    num_train_epochs=4, # 4 epochs is usually the sweet spot for DeBERTa
    warmup_steps=100,
    weight_decay=0.01,

    fp16=False, # Safe standard float16 for speed on T4
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro"
)

# Custom metric function for Trainer
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    accuracy = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')

    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# ---------------------------------------------------------
# 7. EXECUTION
# ---------------------------------------------------------
print("Starting Training...")
trainer.train()

print("\nTraining Complete! Generating Final Evaluation...")
eval_results = trainer.evaluate()

print("\n--- FINAL DEBERTA RESULTS ---")
print(f"Accuracy:    {eval_results['eval_accuracy']:.4f}")
print(f"Macro F1:    {eval_results['eval_f1_macro']:.4f}")
print(f"Weighted F1: {eval_results['eval_f1_weighted']:.4f}")

# Generate detailed classification report
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids
print("\nDetailed Report:\n", classification_report(labels, preds, target_names=list(label2id.keys())))

GPU memory cleared. Seeds locked.
Mounted at /content/drive
Colab environment and Google Drive configured. Model cache at: /content/drive/MyDrive/progettoLLM/hf_cache
Acquiring token from Google Secrets...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face authentication completed.
Loading Tokenizer...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...


Training Complete! Generating Final Evaluation...


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
0.905436,0.868348,4,0.668831,0.267185,0.536106



--- FINAL DEBERTA RESULTS ---
Accuracy:    0.6688
Macro F1:    0.2672
Weighted F1: 0.5361



Detailed Report:
                  precision    recall  f1-score   support

    Clear Reply       0.00      0.00      0.00        79
     Ambivalent       0.67      1.00      0.80       206
Clear Non-Reply       0.00      0.00      0.00        23

       accuracy                           0.67       308
      macro avg       0.22      0.33      0.27       308
   weighted avg       0.45      0.67      0.54       308



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
